In [1]:
import pandas as pd
import numpy as np
import random
import torch
import os
import csv
import time
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
from itertools import combinations
import random
from gensim.models import LdaModel
import spacy

import sys
sys.path.append('./tools')
from Matave import Matave

In [2]:
K_RANGE = list(range(3, 20)) # chosen based on original MATAVE paper use
TOP_N = 10

nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner", "parser"]  # speed
)

In [3]:
# Random States
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
domains = {
    'yahoo': 'non-factoid question',
    'banking77': 'banking text',
    'huffPostNews': 'news',
    'clinc150': 'multi-domain intent',
    'atis': 'air travel information system',
    'medicalAbstracts': 'medical abstract (current patient condition)',
    'dementiaAudio': 'dementia cookie theft picture description',
    'syntheticCareHomeNurseNotes': 'nursing home resident',
    'clinicalDialogueSummarizations': 'clinical note',
    'simSUM': 'compact clinical note'
}

In [6]:
# remove punctuation, make lowercase, remove stopwords, punctuation, lemmatize, remove documents with less than 5 tokens
def preprocess_texts(texts, min_words = 5):
    cleaned_texts = []
    for doc in nlp.pipe(texts, batch_size=1000):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and token.lemma_ != "-PRON-"
            and token.is_alpha
        ]
        if len(tokens) >= min_words:
            cleaned_texts.append(" ".join(tokens))
    return cleaned_texts

In [7]:
def make_lda(corpus, dictionary, matave, tokenized_texts, k):
    start = time.time()

    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=matave.random_state,
        passes=10
    )

    lda_topics = [
        [word for word, _ in lda_model.show_topic(i, topn=TOP_N)]
        for i in range(k)
    ]
    end = time.time()
    coherence = get_coherence_score(lda_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(lda_topics)
    # Inverse redundancy.
    redundancy = compute_topic_redundancy(lda_topics)
    model_time = end - start
    return coherence, diversity, redundancy, model_time, lda_topics, lda_model


In [8]:
parent_path = '../getText/datasetsPrep'
all_results = []
for folder in os.listdir(f'{parent_path}'):
    if os.path.isdir(f'{parent_path}/{folder}'):
        for file in os.listdir(f'{parent_path}/{folder}'):
            if file.endswith('.csv'):
                temp_dataset_name = file.replace('.csv', '')
                df = pd.read_csv(f'{parent_path}/{folder}/{file}')
                df = df.dropna().sample(frac=1, random_state=RANDOM_STATE)
                texts = df['text'].tolist()
                texts = texts[:100]
                texts = preprocess_texts(texts)
                # Run MATAVE algorithm. 
                tokenized_texts = [word_tokenize(text.lower()) for text in texts]
                dictionary = Dictionary(tokenized_texts)
                corpus = [dictionary.doc2bow(text) for text in tokenized_texts]
                start = time.time()
                matave = Matave(texts, random_state = RANDOM_STATE)
                matave.fit(k_range = K_RANGE)
                matave_topics = [topic.split() for topic in matave.top_topic_words.values()]
                matave_topics = [topic[:TOP_N] for topic in matave_topics]
                end = time.time()
                matave_coherence = get_coherence_score(matave_topics, tokenized_texts, dictionary, 'c_v')
                matave_diversity = get_diversity_score(matave_topics)
                matave_redundancy = compute_topic_redundancy(matave_topics)
                matave_model_time = end - start
                topics_for_prompts_dict = {}
                for text, topic in zip(texts, matave.assigned_topics):
                    if topic in topics_for_prompts_dict:
                        topics_for_prompts_dict[topic].append(text)
                    else:
                        topics_for_prompts_dict[topic] = [text]

                matave_example_notes = []
                matave_topic_keywords = []
                for key, items in topics_for_prompts_dict.items():
                    matave_example_notes.append(random.choice(items))
                    matave_topic_keywords.append(f"Other topic keywords: {matave.top_topic_words[key]}")

                for_prompts_dict = {'topic_model': 'MATAVE', 'coherence': matave_coherence, 'diversity': matave_diversity, 'redundancy': matave_redundancy, 'time': matave_model_time,'dataset': temp_dataset_name, 'domain': domains[temp_dataset_name], 'example_notes': '*** SEPARATION ***'.join(matave_example_notes), 'topic_keywords': '\n'.join(matave_topic_keywords)}
                all_results.append(for_prompts_dict)

                # Run LDA algorithm.
                lda_temp = {'k': [], 'coherence': [], 'diversity': [], 'redundancy': [], 'combined': [], 'time': []}
                for k in K_RANGE:

                        lda_coherence, lda_diversity, lda_redundancy, lda_model_time, _, _ = make_lda(corpus, dictionary, matave, tokenized_texts, k)
                        lda_temp['k'].append(k)
                        lda_temp['coherence'].append(lda_coherence)
                        lda_temp['diversity'].append(lda_diversity)
                        lda_temp['redundancy'].append(lda_redundancy)
                        lda_temp['combined'].append((lda_coherence + lda_diversity + lda_redundancy) / 3)
                        lda_temp['time'].append(lda_model_time)

                chosen_k = lda_temp['k'][lda_temp['combined'].index(max(lda_temp['combined']))]
                lda_coherence, lda_diversity, lda_redundancy, lda_model_time, lda_topics, lda_model = make_lda(corpus, dictionary, matave, tokenized_texts, chosen_k)
                doc_topics = [lda_model.get_document_topics(doc) for doc in corpus]
                assigned_topics = [
                    max(topics, key=lambda x: x[1])[0] if topics else None
                    for topics in doc_topics
                ]

                topics_for_prompts_dict = {}
                for text, topic in zip(texts, assigned_topics):
                    if topic in topics_for_prompts_dict:
                        topics_for_prompts_dict[topic].append(text)
                    else:
                        topics_for_prompts_dict[topic] = [text]

                lda_example_notes = []
                lda_topic_keywords = []
                for key, items in topics_for_prompts_dict.items():
                    lda_example_notes.append(random.choice(items))
                    lda_topic_keywords.append(f"Other topic keywords: {' '.join(lda_topics[key])}")

                for_prompts_dict = {'topic_model': 'LDA', 'coherence': lda_coherence, 'diversity': lda_diversity, 'redundancy': lda_redundancy, 'time': lda_model_time,'dataset': temp_dataset_name, 'domain': domains[temp_dataset_name], 'example_notes': '*** SEPARATION ***'.join(lda_example_notes), 'topic_keywords': '\n'.join(lda_topic_keywords)}
                all_results.append(for_prompts_dict)


all_results_df = pd.DataFrame(all_results)
all_results_df.to_csv('./promptDataPreparation.csv', index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', lineterminator='\n')
all_results_df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

,topic_model,coherence,diversity,redundancy,time,dataset,domain,example_notes,topic_keywords
0,MATAVE,0.496931,1.000000,1.000000,6.425270,yahoo,non-factoid question,beat azz year come credit report everyday time...,Other topic keywords: report manual resent mon...
1,LDA,0.409356,0.966667,0.966667,0.135232,yahoo,non-factoid question,slightly misinformed tumor calcify harden turn...,Other topic keywords: like look time good help...
2,MATAVE,0.380064,1.000000,1.000000,2.656240,banking77,banking text,know go charge use card*** SEPARATION ***card ...,Other topic keywords: want tell credit happen ...
3,LDA,0.499766,0.783333,0.893333,0.038817,banking77,banking text,want physical card virtual version*** SEPARATI...,Other topic keywords: card want problem help f...
4,MATAVE,0.467953,0.900000,0.933333,2.375031,huffPostNews,news,wacky food lie parent tell*** SEPARATION ***li...,Other topic keywords: parent food wacky child ...
5,LDA,0.544169,0.975000,0.983333,0.091208,huffPostNews,news,major protest like saturday generally limit ur...,Other topic keywords: like say love woman want...
6,MATAVE,0.302771,0.440909,0.812554,2.867800,clinc150,multi-domain intent,alert bank let know travel brussels*** SEPARAT...,Other topic keywords: card credit discover los...
7,LDA,0.367466,0.960000,0.980000,0.020419,clinc150,multi-domain intent,alert bank let know travel brussels*** SEPARAT...,Other topic keywords: travel know let beach vi...
8,MATAVE,0.394933,0.680000,0.924444,2.515166,atis,air travel information system,list flight seattle continental depart pm*** S...,Other topic keywords: seattle continental chic...
9,LDA,0.329617,0.725000,0.750000,0.101373,atis,air travel information system,cheap fare atlanta san francisco*** SEPARATION...,Other topic keywords: flight san francisco den...
